# 01 — Exploratory data analysis

Run `python -m src.ingest --raw_dir ../data/raw --out ../data/processed` first.

This notebook covers:
- review length distribution (whitespace tokens)
- 1-10 rating distribution and the negative/neutral/positive bucketization
- top conditions by review count
- top drugs by review count
- temporal patterns of review volume and mean rating

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('../data/processed/clean.parquet')
df = pd.read_parquet(DATA)
print(df.shape)
df.head()

In [ ]:
# review length
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['n_tokens'].clip(upper=400).hist(bins=50, ax=ax[0])
ax[0].set_title('Review length (tokens, clipped at 400)')
df['rating'].astype(int).value_counts().sort_index().plot.bar(ax=ax[1])
ax[1].set_title('Rating distribution (1-10)')
plt.tight_layout()

In [ ]:
# sentiment 3-class balance
df['sentiment_3'].value_counts(normalize=True).round(3)

In [ ]:
# top 25 conditions and drugs
top_cond = df['condition'].dropna().value_counts().head(25)
top_drug = df['drug'].dropna().value_counts().head(25)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
top_cond[::-1].plot.barh(ax=ax[0]); ax[0].set_title('Top 25 conditions')
top_drug[::-1].plot.barh(ax=ax[1]); ax[1].set_title('Top 25 drugs')
plt.tight_layout()

In [ ]:
# temporal
if df['date'].notna().any():
    monthly = (df.dropna(subset=['date'])
                 .assign(ym=lambda x: x['date'].dt.to_period('M').dt.to_timestamp())
                 .groupby('ym')
                 .agg(n=('review_id', 'count'), mean_rating=('rating', 'mean')))
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax2 = ax1.twinx()
    ax1.bar(monthly.index, monthly['n'], width=20, color='#cce5ff', label='n reviews')
    ax2.plot(monthly.index, monthly['mean_rating'], color='#cc0000', label='mean rating')
    ax1.set_ylabel('reviews / month'); ax2.set_ylabel('mean rating')
    plt.title('Review volume and mean rating over time')
else:
    print('No dates available (Druglib has no date column).')